### 전처리 최종본

**튜터 피드백 전체 반영**

수정 사항:
1. merge_asof backward join으로 impedance 매핑 방식 변경 — cycle 번호 근사 매핑 제거하고 start_time 기준으로 현재 시점 이전 가장 최근 impedance 값만 사용하도록 수정.
2. impedance_available 플래그 추가 — 첫 impedance 측정 이전/이후 구간을 명확히 구분해서 결측 종류를 분리.
3. 조건부 ffill 적용 — 이전 구간은 NaN 유지(과거 정보 없음), 이후 구간만 마지막 cumulative mean으로 ffill. 

# 1. 라이브러리 Import

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("=" * 60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 2. 데이터 로드

- metadata.csv 파일을 불러오는 첫 단계
- 총 7,565행, 34개 배터리 데이터
- charge (충전), discharge (방전), impedance (내부저항)

In [2]:
import pandas as pd
import os

base_path = "data"
meta_path = os.path.join(base_path, "metadata.csv")

df = pd.read_csv(meta_path)

print(df.shape)
df.head()

(7565, 10)


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


# 3. start_time 전처리

In [3]:
def clear_time(x):
    if pd.isna(x):
        return pd.NaT

    x = str(x)
    nums = []
    current = ''

    for ch in x:
        if ch.isdigit() or ch in ['.', 'e', 'E', '+', '-']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''

    if current != '':
        nums.append(current)

    if len(nums) < 6:
        return pd.NaT

    try:
        nums = list(map(float, nums[:6]))
    except:
        return pd.NaT

    year, month, day, hour, minute, second = nums

    if not (2000 <= year <= 2100 and
            1 <= month <= 12 and
            1 <= day <= 31 and
            0 <= hour < 24 and
            0 <= minute < 60 and
            0 <= second < 60):
        return pd.NaT

    try:
        return pd.Timestamp(
            int(year), int(month), int(day),
            int(hour), int(minute), int(second)
        )
    except:
        return pd.NaT


df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])

### 3-1. 시간 파생 칼럼 생성

- `elapsed_hours_from_first`: 배터리별 첫 가동 후 경과 시간
- `hours_from_prev_operation`: 직전 작업 시작시각과의 차이
  - 주의: 순수 휴지시간이 아니라 charge·discharge·impedance 포함한 직전 작업 시작시각과의 차이 (휴지기 + 작업시간 혼재)

In [4]:
# 1. 시간 순으로 정렬
df = df.sort_values(by=['battery_id', 'start_time']).reset_index(drop=True)

# 2. elapsed_hours_from_first: 배터리별 첫 가동 후 경과 시간
df['elapsed_hours_from_first'] = df.groupby('battery_id')['start_time'].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 3600
)

# 3. hours_from_prev_operation 
# 주의: 직전 discharge 이후 순수 휴지시간이 아니라
#       charge·discharge·impedance 포함한 직전 작업 시작시각과의 차이
#       (휴지기 + 작업시간 혼재)
df['hours_from_prev_operation'] = (
    df.groupby('battery_id')['start_time'].diff().dt.total_seconds() / 3600
)
df['hours_from_prev_operation'] = df['hours_from_prev_operation'].fillna(0)

# 4. 그룹 컬럼 추가

- group: 실험 조건 그룹 (A~I)
- end_reason: 실험 종료 사유
- analysis_role: 분석 역할

In [5]:
group_map = {
    'B0005':'A','B0006':'A','B0007':'A','B0018':'A',
    'B0025':'B','B0026':'B','B0027':'B','B0028':'B',
    'B0029':'C','B0030':'C','B0031':'C','B0032':'C',
    'B0033':'D','B0034':'D','B0036':'D',
    'B0038':'E','B0039':'E','B0040':'E',
    'B0041':'F','B0042':'F','B0043':'F','B0044':'F',
    'B0045':'G','B0046':'G','B0047':'G','B0048':'G',
    'B0049':'H','B0050':'H','B0051':'H','B0052':'H',
    'B0053':'I','B0054':'I','B0055':'I','B0056':'I',
}

end_reason_map = {
    'A': 'EOL',      'B': 'censored', 'C': 'censored',
    'D': 'QA_issue', 'E': 'QA_issue', 'F': 'QA_issue',
    'G': 'QA_issue', 'H': 'crashed',  'I': 'QA_issue',
}

group_info = {
    'A': {'온도':'실온',     '방전전류':'2A CC',      'cutoff':'battery별 상이', '신뢰도':'정상'},
    'B': {'온도':'24°C',    '방전전류':'4A 스퀘어파', 'cutoff':'2.0~2.7V',       '신뢰도':'주의'},
    'C': {'온도':'43°C',    '방전전류':'4A',          'cutoff':'2.0~2.7V',       '신뢰도':'주의'},
    'D': {'온도':'24°C',    '방전전류':'4A/2A 혼합',  'cutoff':'2.0~2.7V',       '신뢰도':'비정상'},
    'E': {'온도':'24&44°C', '방전전류':'1/2/4A 복합', 'cutoff':'2.2~2.7V',       '신뢰도':'비정상'},
    'F': {'온도':'4°C',     '방전전류':'4A/1A 혼합',  'cutoff':'2.0~2.7V',       '신뢰도':'비정상'},
    'G': {'온도':'4°C',     '방전전류':'1A',          'cutoff':'2.0~2.7V',       '신뢰도':'비정상'},
    'H': {'온도':'4°C',     '방전전류':'2A',          'cutoff':'2.0~2.7V',       '신뢰도':'심각'},
    'I': {'온도':'4°C',     '방전전류':'2A',          'cutoff':'2.0~2.7V',       '신뢰도':'비정상'},
}

analysis_role_map = {
    'A': 'main',    'B': 'excluded',  'C': 'comparison',
    'D': 'excluded','E': 'excluded',  'F': 'anomaly',
    'G': 'anomaly', 'H': 'anomaly',   'I': 'anomaly',
}

df['group']         = df['battery_id'].map(group_map)
df['end_reason']    = df['group'].map(end_reason_map)
df['analysis_role'] = df['group'].map(analysis_role_map)

print("\n그룹별 배터리 수 및 역할:")
summary = df.groupby(['group','end_reason','analysis_role'])['battery_id'].nunique().reset_index()
summary.columns = ['group','end_reason','analysis_role','배터리수']
print(summary.to_string(index=False))


그룹별 배터리 수 및 역할:
group end_reason analysis_role  배터리수
    A        EOL          main     4
    B   censored      excluded     4
    C   censored    comparison     4
    D   QA_issue      excluded     3
    E   QA_issue      excluded     3
    F   QA_issue       anomaly     4
    G   QA_issue       anomaly     4
    H    crashed       anomaly     4
    I   QA_issue       anomaly     4


# 5. 실험 조건 라벨 & battery_protocol_map

- [수정 2] `test_temperature_profile`: 원본 `ambient_temperature`와 분리된 해석용 라벨
- [수정 4] `battery_protocol_map`: cutoff_voltage를 battery_id 기준으로 관리

In [6]:
# ambient_temperature: 원본 row-level 관측값 (건드리지 않음)
#           test_temperature_profile: 해석용 라벨 (그룹 기준)
# F그룹: 실제 데이터에 4°C와 22°C 혼재 확인

test_temperature_profile_map = {
    'A': '24°C_stable',    'B': '24°C_stable',
    'C': '43°C_stable',    'D': '24°C_stable',
    'E': '24_44°C_mixed',
    'F': '4°C_22°C_mixed',  # 실제 데이터에 4°C와 22°C 혼재 확인
    'G': '4°C_stable',    'H': '4°C_stable',    'I': '4°C_stable',
}

load_profile_map = {
    'A': '2A_CC',         'B': '4A_squarewave', 'C': '4A_CC',
    'D': '2A_4A_mixed',   'E': '1A_2A_4A_mixed','F': '4A_1A_mixed',
    'G': '1A_CC',         'H': '2A_CC',         'I': '2A_CC',
}

eol_rule_source_map = {
    'A': 'NASA_30%fade', 'B': 'censored',      'C': 'censored',
    'D': 'NASA_20%fade', 'E': 'NASA_20%fade',
    'F': 'NASA_30%fade', 'G': 'NASA_30%fade',
    'H': 'crashed',      'I': 'NASA_30%fade',
}

df['test_temperature_profile'] = df['group'].map(test_temperature_profile_map)
df['load_profile']             = df['group'].map(load_profile_map)
df['eol_rule_source']          = df['group'].map(eol_rule_source_map)

In [7]:
# battery_protocol_map
# 튜터 피드백: "group_map 말고 battery_protocol_map을 따로 두는 게 맞습니다"
# cutoff_voltage를 battery_id 기준으로 관리
# 출처: NASA README

battery_protocol_map = {
    # 그룹A: 상온 2A CC — battery별 cutoff 다름
    'B0005': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0006': {'cutoff_voltage': 2.5, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0007': {'cutoff_voltage': 2.2, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0018': {'cutoff_voltage': 2.5, 'discharge_current': '2A',            'eol_fade': 0.30},
    # 그룹B
    'B0025': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0026': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0027': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0028': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    # 그룹C
    'B0029': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0030': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0031': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0032': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    # 그룹D
    'B0033': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    'B0034': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    'B0036': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    # 그룹E
    'B0038': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    'B0039': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    'B0040': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    # 그룹F
    'B0041': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0042': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0043': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0044': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    # 그룹G
    'B0045': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0046': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0047': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0048': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    # 그룹H: SW 크래시 — EOL 기준 없음
    'B0049': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0050': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0051': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0052': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    # 그룹I
    'B0053': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0054': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0055': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0056': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
}

df['cutoff_voltage']     = df['battery_id'].map({k: v['cutoff_voltage']    for k, v in battery_protocol_map.items()})
df['discharge_current']  = df['battery_id'].map({k: v['discharge_current'] for k, v in battery_protocol_map.items()})
df['eol_fade_threshold'] = df['battery_id'].map({k: v['eol_fade']          for k, v in battery_protocol_map.items()})

print("=" * 55)
print("battery_protocol_map 적용 결과")
print("=" * 55)
protocol_summary = (
    df.groupby('battery_id')
    .agg(
        group             = ('group',             'first'),
        cutoff_voltage    = ('cutoff_voltage',    'first'),
        discharge_current = ('discharge_current', 'first'),
        eol_fade_threshold= ('eol_fade_threshold','first'),
    )
    .reset_index()
)
print(protocol_summary.to_string(index=False))

battery_protocol_map 적용 결과
battery_id group  cutoff_voltage discharge_current  eol_fade_threshold
     B0005     A             2.7                2A                 0.3
     B0006     A             2.5                2A                 0.3
     B0007     A             2.2                2A                 0.3
     B0018     A             2.5                2A                 0.3
     B0025     B             2.7     4A_squarewave                 0.3
     B0026     B             2.7     4A_squarewave                 0.3
     B0027     B             2.7     4A_squarewave                 0.3
     B0028     B             2.7     4A_squarewave                 0.3
     B0029     C             2.7                4A                 0.3
     B0030     C             2.7                4A                 0.3
     B0031     C             2.7                4A                 0.3
     B0032     C             2.7                4A                 0.3
     B0033     D             2.0       4A_2A_mixed

# 6. 타입별 3개 테이블 분리

- charge·discharge·impedance를 분리
- 전역 dropna() 금지 (구조적 결측 존재)

In [8]:
df_charge    = df[df['type'] == 'charge'].copy()
df_discharge = df[df['type'] == 'discharge'].copy()
df_impedance = df[df['type'] == 'impedance'].copy()

print("\n" + "=" * 55)
print("타입별 분리 완료")
print("=" * 55)
print(f"df_charge    : {len(df_charge):,}행")
print(f"df_discharge : {len(df_discharge):,}행")
print(f"df_impedance : {len(df_impedance):,}행")


타입별 분리 완료
df_charge    : 2,815행
df_discharge : 2,794행
df_impedance : 1,956행


# 7. Capacity 상태 flag 추가

- valid / zero / missing / low_anomaly / impossible_high

In [9]:
df_discharge['Capacity'] = pd.to_numeric(df_discharge['Capacity'], errors='coerce')

def classify_capacity(cap):
    if pd.isna(cap):   return 'missing'
    elif cap == 0:     return 'zero'
    elif cap > 2.1:    return 'impossible_high'
    elif cap < 0.3:    return 'low_anomaly'
    else:              return 'valid'

df_discharge['cap_flag']   = df_discharge['Capacity'].apply(classify_capacity)

hard_exclude = ['missing', 'zero', 'impossible_high']
df_discharge['cap_exclude'] = df_discharge['cap_flag'].isin(hard_exclude)
df_discharge['cap_anomaly'] = df_discharge['cap_flag'] == 'low_anomaly'

print("\n" + "=" * 55)
print("Capacity flag 분포 (전체)")
print("=" * 55)
print(df_discharge['cap_flag'].value_counts())
print("\n하드 제외 후보:", df_discharge['cap_exclude'].sum())
print("이상탐지 대상:", df_discharge['cap_anomaly'].sum())

print("\n그룹별 flag 분포:")
flag_grp = (
    df_discharge.groupby(['group','cap_flag'])
    .size().unstack(fill_value=0).reset_index()
)
print(flag_grp.to_string(index=False))


Capacity flag 분포 (전체)
cap_flag
valid              2553
low_anomaly         193
missing              25
zero                 19
impossible_high       4
Name: count, dtype: int64

하드 제외 후보: 48
이상탐지 대상: 193

그룹별 flag 분포:
group  impossible_high  low_anomaly  missing  valid  zero
    A                0            0        0    636     0
    B                0            0        0    112     0
    C                0            0        0    160     0
    D                1            6        0    584     0
    E                0            2        0    139     0
    F                0          180        0    220     3
    G                0            0        0    277    11
    H                3            5       25     64     3
    I                0            0        0    361     2


### 7-1. Impedance 상태 flag 추가

- valid / missing / zero_or_minus / high_anomaly / noise_candidate / rct_imbalance

In [10]:
df_impedance['Re']  = pd.to_numeric(df_impedance['Re'],  errors='coerce')
df_impedance['Rct'] = pd.to_numeric(df_impedance['Rct'], errors='coerce')
df_impedance = df_impedance.sort_values(['battery_id', 'start_time']).reset_index(drop=True)

df_impedance['re_diff'] = df_impedance.groupby('battery_id')['Re'].diff().abs()

def classify_impedance(row):
    re, rct, diff = row['Re'], row['Rct'], row['re_diff']
    if pd.isna(re) or pd.isna(rct):   return 'missing'
    if re <= 0 or rct <= 0:           return 'zero_or_minus'
    if pd.notna(diff) and diff > 0.3: return 'noise_candidate'
    if rct > re * 10:                 return 'rct_imbalance'
    if re > 1.0 or rct > 1.5:        return 'high_anomaly'
    return 'valid'

df_impedance['imp_flag'] = df_impedance.apply(classify_impedance, axis=1)

hard_exclude_imp = ['missing', 'zero_or_minus']
df_impedance['imp_exclude'] = df_impedance['imp_flag'].isin(hard_exclude_imp)

anomaly_imp = ['high_anomaly', 'noise_candidate', 'rct_imbalance']
df_impedance['imp_anomaly'] = df_impedance['imp_flag'].isin(anomaly_imp)

print("=" * 55)
print("Impedance Flag 분포 결과")
print("=" * 55)
print(df_impedance['imp_flag'].value_counts())
print(f"\n하드 제외 대상: {df_impedance['imp_exclude'].sum()}건")
print(f"이상 탐지 대상: {df_impedance['imp_anomaly'].sum()}건")

imp_summary = df_impedance.groupby(['group','imp_flag']).size().unstack(fill_value=0)
print("\n[그룹별 상세 분포]")
print(imp_summary)

Impedance Flag 분포 결과
imp_flag
valid              1932
zero_or_minus        11
missing               9
noise_candidate       4
Name: count, dtype: int64

하드 제외 대상: 20건
이상 탐지 대상: 4건

[그룹별 상세 분포]
imp_flag  missing  noise_candidate  valid  zero_or_minus
group                                                   
A               0                0    887              0
B               0                0     84              0
C               0                0     68              0
D               0                0    276              0
E               0                0     84              0
F               0                0    179              0
G               0                0    160              0
H               9                4     24             11
I               0                0    170              0


# 8. discharge 사이클 순번 추가

- `discharge_cycle_raw`: 전체 기록 기준 순번
- `discharge_cycle_valid`: hard exclusion 제외 후 순번

In [11]:
df_discharge = df_discharge.sort_values(
    ['battery_id', 'start_time', 'filename']
).reset_index(drop=True)

# 전체 discharge 시도 기준 순번 (이상값 포함)
df_discharge['discharge_cycle_raw'] = (
    df_discharge.groupby('battery_id').cumcount() + 1
)

# hard exclusion 여부
df_discharge['is_hard_excluded'] = df_discharge['cap_flag'].isin(
    ['missing', 'zero', 'impossible_high']
)

# valid discharge 순번 (hard exclusion 제외)
df_discharge['discharge_cycle_valid'] = (
    df_discharge.loc[~df_discharge['is_hard_excluded']]
    .groupby('battery_id')
    .cumcount() + 1
)

# 9. 초기 Capacity 계산

- valid 구간 첫 5개 중앙값 사용
- 첫 번째 사이클 단독 사용 금지

In [12]:
init_cap = (
    df_discharge[df_discharge['cap_flag'] == 'valid']
    .groupby('battery_id')['Capacity']
    .apply(lambda x: x.head(5).median())
    .rename('init_cap')
)

df_discharge = df_discharge.merge(init_cap, on='battery_id', how='left')

print("\n" + "=" * 55)
print("초기 Capacity (valid 구간 첫 5개 중앙값)")
print("=" * 55)
init_summary = df_discharge.groupby(['battery_id','group'])['init_cap'].first().reset_index()
print(init_summary.round(4).to_string(index=False))


초기 Capacity (valid 구간 첫 5개 중앙값)
battery_id group  init_cap
     B0005     A    1.8353
     B0006     A    2.0133
     B0007     A    1.8807
     B0018     A    1.8396
     B0025     B    1.8471
     B0026     B    1.8143
     B0027     B    1.8142
     B0028     B    1.7976
     B0029     C    1.8158
     B0030     C    1.7518
     B0031     C    1.8044
     B0032     C    1.8655
     B0033     D    1.2529
     B0034     D    1.6207
     B0036     D    1.8011
     B0038     E    1.0613
     B0039     E    0.4711
     B0040     E    0.7796
     B0041     F    1.1195
     B0042     F    1.7282
     B0043     F    1.6815
     B0044     F    1.6534
     B0045     G    0.8852
     B0046     G    1.5031
     B0047     G    1.5081
     B0048     G    1.4989
     B0049     H    1.3644
     B0050     H    1.5518
     B0051     H    1.2039
     B0052     H    1.3611
     B0053     I    1.1306
     B0054     I    1.0960
     B0055     I    1.2573
     B0056     I    1.2974


# 10. SOH 계산

- `SOH_nominal`: 공칭 용량(2.0Ah) 기준
- `SOH_relative`: 배터리 자신의 초기값 기준

In [13]:
df_discharge['SOH_nominal'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / 2.0 * 100).round(2),
    np.nan
)

# 주석 추가
# SOH_relative가 초반에 100%를 소폭 초과하는 경우 있음
# 이유: init_cap = valid 첫 5개 중앙값으로 계산했기 때문에
#       초반 측정값이 중앙값보다 살짝 높으면 100% 초과 가능
# → 계산 오류 아님. 정상적인 구조적 특성.
df_discharge['SOH_relative'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / df_discharge['init_cap'] * 100).round(2),
    np.nan
)

In [14]:
df_discharge[["battery_id","Capacity","SOH_nominal","SOH_relative"]].head(20)

,battery_id,Capacity,SOH_nominal,SOH_relative
0,B0005,1.856487,92.82,101.15
1,B0005,1.846327,92.32,100.60
2,B0005,1.835349,91.77,100.00
3,B0005,1.835263,91.76,100.00
4,B0005,1.834646,91.73,99.96
5,B0005,1.835662,91.78,100.02
6,B0005,1.835146,91.76,99.99
7,B0005,1.825757,91.29,99.48
8,B0005,1.824774,91.24,99.42
9,B0005,1.824613,91.23,99.42


# 11. EOL 탐지 & RUL 계산

- [수정 6] `rul_label_type` 추가: main만 supervised로 분리
- 튜터 피드백: excluded·anomaly까지 eol_cycle 계산하면 analysis_role과 충돌

In [15]:
for col in ['eol_cycle', 'RUL', 'eol_soh_threshold']:
    if col in df_discharge.columns:
        df_discharge = df_discharge.drop(columns=col)

eol_threshold_map = {
    'A': 70, 'B': 70, 'C': 70,
    'D': 80, 'E': 80,
    'F': 70, 'G': 70,
    'H': None,
    'I': 70
}

df_discharge['eol_soh_threshold'] = df_discharge['group'].map(eol_threshold_map)

eol_cycles = (
    df_discharge[
        (df_discharge['cap_flag'] == 'valid') &
        (df_discharge['eol_soh_threshold'].notna()) &
        (df_discharge['SOH_nominal'] < df_discharge['eol_soh_threshold'])
    ]
    .groupby('battery_id')['discharge_cycle_raw']
    .min()
    .rename('eol_cycle')
    .reset_index()
)

df_discharge = df_discharge.merge(eol_cycles, on='battery_id', how='left')

df_discharge['RUL'] = np.where(
    df_discharge['eol_cycle'].notna(),
    (df_discharge['eol_cycle'] - df_discharge['discharge_cycle_raw']).clip(lower=0),
    np.nan
)

# rul_label_type 추가
# main만 정식 ML 학습용 supervised 라벨
# 나머지는 용도별로 분리 (analysis_role과 충돌 방지)
df_discharge['rul_label_type'] = np.where(
    df_discharge['analysis_role'] == 'main',       'supervised',
np.where(
    df_discharge['analysis_role'] == 'comparison', 'censored',
np.where(
    df_discharge['analysis_role'] == 'anomaly',    'anomaly_case',
    'unsupported_for_rul'
)))

print("\n" + "=" * 55)
print("전체 그룹 EOL & RUL 요약")
print("=" * 55)

eol_summary = (
    df_discharge.groupby('battery_id')
    .agg(
        group          = ('group',             'first'),
        analysis_role  = ('analysis_role',     'first'),
        end_reason     = ('end_reason',        'first'),
        rul_label_type = ('rul_label_type',    'first'),
        eol_threshold  = ('eol_soh_threshold', 'first'),
        eol_cycle      = ('eol_cycle',         'first'),
        total_cycles   = ('discharge_cycle_raw','max'),
        rul_available  = ('RUL', lambda x: x.notna().sum()),
    )
    .reset_index()
)

eol_summary['eol_달성'] = eol_summary['eol_cycle'].notna().map({True: 'O', False: 'X'})
print(eol_summary.to_string(index=False))

print("\nrul_label_type 분포:")
print(df_discharge['rul_label_type'].value_counts())


전체 그룹 EOL & RUL 요약
battery_id group analysis_role end_reason      rul_label_type  eol_threshold  eol_cycle  total_cycles  rul_available eol_달성
     B0005     A          main        EOL          supervised           70.0      125.0           168            168      O
     B0006     A          main        EOL          supervised           70.0      109.0           168            168      O
     B0007     A          main        EOL          supervised           70.0        NaN           168              0      X
     B0018     A          main        EOL          supervised           70.0       97.0           132            132      O
     B0025     B      excluded   censored unsupported_for_rul           70.0        NaN            28              0      X
     B0026     B      excluded   censored unsupported_for_rul           70.0        6.0            28             28      O
     B0027     B      excluded   censored unsupported_for_rul           70.0        NaN            28           

# 11-1. rul_subset 생성

- main 그룹에서만 공식 RUL 계산

In [16]:
rul_subset = eol_summary[eol_summary['analysis_role'] == 'main']

rul_ids = rul_subset['battery_id']
rul_df  = df_discharge[df_discharge['battery_id'].isin(rul_ids)].copy()
rul_df['RUL'] = rul_df['eol_cycle'] - rul_df['discharge_cycle_raw']

In [17]:
valid_cap = (
    df_discharge[df_discharge['cap_flag'] == 'valid']
    .groupby('battery_id')['Capacity']
    .min().reset_index()
    .rename(columns={'Capacity': 'min_cap'})
)

rul_count = (
    df_discharge.groupby('battery_id')['RUL']
    .count().reset_index()
    .rename(columns={'RUL': 'rul_count'})
)

eol_summary1 = (
    df_discharge.groupby('battery_id')
    .agg(
        group          = ('group',              'first'),
        analysis_role  = ('analysis_role',      'first'),
        end_reason     = ('end_reason',         'first'),
        rul_label_type = ('rul_label_type',     'first'),
        eol_cycle      = ('eol_cycle',          'max'),
        total_cycles   = ('discharge_cycle_raw','max')
    )
    .reset_index()
)

eol_summary1 = (
    eol_summary1
    .merge(valid_cap, on='battery_id', how='left')
    .merge(rul_count, on='battery_id', how='left')
)
eol_summary1['eol_달성'] = eol_summary1['eol_cycle'].notna().map({True:'O', False:'X'})
print(eol_summary1.to_string(index=False))

battery_id group analysis_role end_reason      rul_label_type  eol_cycle  total_cycles  min_cap  rul_count eol_달성
     B0005     A          main        EOL          supervised      125.0           168 1.287453        168      O
     B0006     A          main        EOL          supervised      109.0           168 1.153818        168      O
     B0007     A          main        EOL          supervised        NaN           168 1.400455          0      X
     B0018     A          main        EOL          supervised       97.0           132 1.341051        132      O
     B0025     B      excluded   censored unsupported_for_rul        NaN            28 1.767789          0      X
     B0026     B      excluded   censored unsupported_for_rul        6.0            28 1.386337         28      O
     B0027     B      excluded   censored unsupported_for_rul        NaN            28 1.770093          0      X
     B0028     B      excluded   censored unsupported_for_rul        NaN            28 1

# 12. 그룹별 DataFrame 분리

In [18]:
df_main       = df_discharge[df_discharge['analysis_role'] == 'main'].copy()
df_comparison = df_discharge[df_discharge['analysis_role'] == 'comparison'].copy()
df_anomaly    = df_discharge[df_discharge['analysis_role'] == 'anomaly'].copy()
df_excluded   = df_discharge[df_discharge['analysis_role'] == 'excluded'].copy()

# 13. impedance 전처리 및 역할별 분리

In [19]:
df_impedance['group']         = df_impedance['battery_id'].map(group_map)
df_impedance['end_reason']    = df_impedance['group'].map(end_reason_map)
df_impedance['analysis_role'] = df_impedance['group'].map(analysis_role_map)
df_impedance = df_impedance.sort_values(['battery_id','test_id'])
df_impedance['impedance_cycle_no'] = (
    df_impedance.groupby('battery_id').cumcount() + 1
)

df_imp_main       = df_impedance[df_impedance['analysis_role'] == 'main'].copy()
df_imp_comparison = df_impedance[df_impedance['analysis_role'] == 'comparison'].copy()
df_imp_anomaly    = df_impedance[df_impedance['analysis_role'] == 'anomaly'].copy()

group_dfs = {}
for grp in sorted(df_discharge['group'].unique()):
    group_dfs[grp] = df_discharge[df_discharge['group'] == grp].copy()

print("\nimpedance 전처리 완료:")
print(f"df_imp_main       : {len(df_imp_main):>4}행")
print(f"df_imp_comparison : {len(df_imp_comparison):>4}행")
print(f"df_imp_anomaly    : {len(df_imp_anomaly):>4}행")

print("\n" + "=" * 55)
print("역할별 DataFrame 분리 결과")
print("=" * 55)
print(f"df_main       (그룹A)    : {len(df_main):>4}행  {df_main['battery_id'].nunique()}개 배터리")
print(f"df_comparison (그룹C)    : {len(df_comparison):>4}행  {df_comparison['battery_id'].nunique()}개 배터리")
print(f"df_anomaly    (F·G·H·I) : {len(df_anomaly):>4}행  {df_anomaly['battery_id'].nunique()}개 배터리")
print(f"df_excluded   (B·D·E)   : {len(df_excluded):>4}행  {df_excluded['battery_id'].nunique()}개 배터리")


impedance 전처리 완료:
df_imp_main       :  887행
df_imp_comparison :   68행
df_imp_anomaly    :  557행

역할별 DataFrame 분리 결과
df_main       (그룹A)    :  636행  4개 배터리
df_comparison (그룹C)    :  160행  4개 배터리
df_anomaly    (F·G·H·I) : 1154행  16개 배터리
df_excluded   (B·D·E)   :  844행  10개 배터리


# 14. ML용 데이터셋 생성

- [수정 7] Re_mean/Rct_mean: cumulative mean으로 변경 (누수 제거)
- [수정 8] battery_id: GroupKFold용으로만 유지 (feature 제외)
- forward fill로 결측치 처리

In [20]:
# Re_mean, Rct_mean을 전체 수명 평균 대신
# 현재 사이클 시점까지 관측된 누적 평균(cumulative mean)으로 계산
# → 미래 impedance 정보 누수 방지

imp_A = df_imp_main[['battery_id','impedance_cycle_no','Re','Rct']].copy()
imp_A = imp_A.sort_values(['battery_id','impedance_cycle_no'])

# impedance cycle → discharge cycle 근사 매핑
imp_A['discharge_cycle_raw'] = ((imp_A['impedance_cycle_no'] + 1) // 2)

# 현재 시점까지 누적 평균 계산
imp_A['Re_cumean'] = (
    imp_A.groupby('battery_id')['Re']
    .expanding().mean().reset_index(level=0, drop=True)
)
imp_A['Rct_cumean'] = (
    imp_A.groupby('battery_id')['Rct']
    .expanding().mean().reset_index(level=0, drop=True)
)

# discharge cycle 기준으로 집계
re_rct_cumean = (
    imp_A.groupby(['battery_id','discharge_cycle_raw'])
    .agg(Re_mean=('Re_cumean','last'), Rct_mean=('Rct_cumean','last'))
    .reset_index()
)

# battery_id는 GroupKFold split 용도로만 유지
# feature로 사용하지 않음 
df_ml = (
    df_main[df_main['cap_flag'] == 'valid']
    [['battery_id','group','discharge_cycle_raw',
      'discharge_cycle_valid','SOH_relative','SOH_nominal',
      'ambient_temperature','RUL','rul_label_type']]
    .dropna(subset=['RUL'])
    .query("rul_label_type == 'supervised'")
    .merge(re_rct_cumean, on=['battery_id','discharge_cycle_raw'], how='left')
    .copy()
)

# 결측치 처리: forward fill
# 이유: impedance 측정이 discharge보다 일찍 끝나서 마지막 구간 NaN 발생
# forward fill = 마지막으로 관측된 cumulative mean 값 유지
df_ml = df_ml.sort_values(['battery_id','discharge_cycle_raw'])
df_ml['Re_mean']  = df_ml.groupby('battery_id')['Re_mean'].ffill()
df_ml['Rct_mean'] = df_ml.groupby('battery_id')['Rct_mean'].ffill()

print("\n" + "=" * 55)
print("ML용 데이터셋")
print("=" * 55)
print(f"총 행 수        : {len(df_ml)}")
print(f"배터리 수       : {df_ml['battery_id'].nunique()}")
print(f"결측치 여부     :\n{df_ml.isnull().sum()}")
print(f"\n샘플 (앞 5행):")
print(df_ml.head().to_string(index=False))


ML용 데이터셋
총 행 수        : 468
배터리 수       : 3
결측치 여부     :
battery_id               0
group                    0
discharge_cycle_raw      0
discharge_cycle_valid    0
SOH_relative             0
SOH_nominal              0
ambient_temperature      0
RUL                      0
rul_label_type           0
Re_mean                  0
Rct_mean                 0
dtype: int64

샘플 (앞 5행):
battery_id group  discharge_cycle_raw  discharge_cycle_valid  SOH_relative  SOH_nominal  ambient_temperature   RUL rul_label_type  Re_mean  Rct_mean
     B0005     A                    1                    1.0        101.15        92.82                   24 124.0     supervised 0.045678  0.072866
     B0005     A                    2                    2.0        100.60        92.32                   24 123.0     supervised 0.045598  0.072059
     B0005     A                    3                    3.0        100.00        91.77                   24 122.0     supervised 0.045581  0.071699
     B0005     A        

# 15. 전체 저장

In [21]:
os.makedirs(base_path, exist_ok=True)

df_discharge.to_csv(os.path.join(base_path, 'df_discharge_processed.csv'), index=False)
df_main.to_csv(os.path.join(base_path, 'df_A_main.csv'), index=False)
df_comparison.to_csv(os.path.join(base_path, 'df_C_comparison.csv'), index=False)
df_anomaly.to_csv(os.path.join(base_path, 'df_anomaly.csv'), index=False)

for grp, gdf in group_dfs.items():
    gdf.to_csv(os.path.join(base_path, f'df_group_{grp}.csv'), index=False)

df_impedance.to_csv(os.path.join(base_path, 'df_impedance_processed.csv'), index=False)
df_imp_main.to_csv(os.path.join(base_path, 'df_imp_A_main.csv'), index=False)
df_imp_comparison.to_csv(os.path.join(base_path, 'df_imp_C_comparison.csv'), index=False)
df_ml.to_csv(os.path.join(base_path, 'df_ml_dataset.csv'), index=False)

print("\n" + "=" * 55)
print("전체 저장 완료")
print("=" * 55)
print("df_discharge_processed.csv  — 전체 34개 배터리 discharge")
print("df_A_main.csv               — 그룹A 메인 분석")
print("df_C_comparison.csv         — 그룹C 온도 비교")
print("df_anomaly.csv              — 그룹F·G·H·I 이상탐지")
print("df_group_{A~I}.csv          — 그룹별 개별 파일")
print("df_impedance_processed.csv  — 전체 impedance")
print("df_imp_A_main.csv           — 그룹A impedance")
print("df_imp_C_comparison.csv     — 그룹C impedance")
print("df_ml_dataset.csv           — ML 학습용 데이터셋")
print("\n다음 단계: 02_EDA.ipynb")


전체 저장 완료
df_discharge_processed.csv  — 전체 34개 배터리 discharge
df_A_main.csv               — 그룹A 메인 분석
df_C_comparison.csv         — 그룹C 온도 비교
df_anomaly.csv              — 그룹F·G·H·I 이상탐지
df_group_{A~I}.csv          — 그룹별 개별 파일
df_impedance_processed.csv  — 전체 impedance
df_imp_A_main.csv           — 그룹A impedance
df_imp_C_comparison.csv     — 그룹C impedance
df_ml_dataset.csv           — ML 학습용 데이터셋

다음 단계: 02_EDA.ipynb
